# 01. Adım Adım Tensör Boyutları ve Akış Analizi

Bu notebook, **"Attention Is All You Need" (Vaswani et al., 2017)** mimarisindeki bir girdinin
katman katman ilerlerken geçirdiği şekil (shape) dönüşümlerini interaktif olarak inceler.

İncelenecek Adımlar:
1. Token ID -> Gömme Tensörü ($d_{model}$ ölçeklemesi)
2. Sinüzoidal Pozisyon Eklemesi ($X + PE$)
3. Multi-Head Projeksiyonları ($Q, K, V$ transpozisyonları)
4. Causal ve Padding Maskeleme ile Dikkat Haritası
5. Encoder Çıktısının Decoder Cross-Attention Katmanına Akışı

In [ ]:
import sys
sys.path.append("..")
import torch
from src import Transformer, create_masks

# 1. Temel Boyutlar
batch_size = 2
seq_len_src = 6
seq_len_tgt = 5
d_model = 512
num_heads = 8
src_vocab_size = 100
tgt_vocab_size = 100

print(f"Batch Boyutu: {batch_size}")
print(f"Kaynak Dizi: {seq_len_src} token | Hedef Dizi: {seq_len_tgt} token")
print(f"Model Boyutu (d_model): {d_model} | Baş Sayısı: {num_heads} | d_k = d_v: {d_model // num_heads}")

## 2. Girdi Tensörleri ve Maskelerin İncelenmesi

In [ ]:
# Rastgele token indeksleri oluşturma
torch.manual_seed(42)
src = torch.randint(1, src_vocab_size, (batch_size, seq_len_src))
tgt = torch.randint(1, tgt_vocab_size, (batch_size, seq_len_tgt))

# Dolgu (Padding) ekleyelim (0: PAD)
src[0, -2:] = 0  # 1. örneğin son 2 tokenı PAD
tgt[1, -1] = 0   # 2. örneğin son 1 tokenı PAD

src_mask, tgt_mask = create_masks(src, tgt, pad_idx=0)

print("Kaynak Dizi (src):\n", src)
print("\nKaynak Maske Şekli (src_mask):", src_mask.shape)
print("Hedef Maske Şekli (tgt_mask):", tgt_mask.shape)

## 3. Kodlayıcı (Encoder) İleri Besleme Adımları

In [ ]:
model = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    num_encoder_layers=6,
    num_decoder_layers=6,
    d_ff=2048,
    dropout=0.1
)
model.eval()

# Kaynak kodlama
memory = model.encode(src, src_mask=src_mask)
print("Encoder Çıktısı (Memory) Şekli:", memory.shape)
assert memory.shape == (batch_size, seq_len_src, d_model)

## 4. Kod Çözücü (Decoder) ve Logit Üretimi

In [ ]:
# Hedef diziyi çözme
decoder_out = model.decode(tgt, memory, src_mask=src_mask, tgt_mask=tgt_mask)
print("Decoder Çıktısı Şekli:", decoder_out.shape)

# Kelime haznesi projeksiyonu (Generator)
logits = model.generator(decoder_out)
print("Nihai Logitler Şekli:", logits.shape)
assert logits.shape == (batch_size, seq_len_tgt, tgt_vocab_size)
print("\nTensör boyut akışı başarıyla doğrulandı!")